In [9]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN__TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [10]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="qwen3:8b", base_url="http://localhost:11434")

In [11]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
llm.invoke([HumanMessage(content="Hello! My name is Aakash and i am a RAG Consultant")])

AIMessage(content="Hello, Aakash! Welcome to the conversation. It's great to meet someone with expertise in RAG (Retrieval-Augmented Generation) consulting. I'm here to help with any questions, ideas, or challenges you might have related to RAG systems, their implementation, optimization, or application in various domains. How are you doing today, and what would you like to discuss or explore? 😊", additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-10-28T10:25:13.132694709Z', 'done': True, 'done_reason': 'stop', 'total_duration': 34270498481, 'load_duration': 1301545, 'prompt_eval_count': 24, 'prompt_eval_duration': 615438000, 'eval_count': 174, 'eval_duration': 33624250999, 'logprobs': None, 'model_name': 'qwen3:8b'}, id='run--01a1820a-59cb-7121-89ee-35966ce8a8d7-0', usage_metadata={'input_tokens': 24, 'output_tokens': 174, 'total_tokens': 198})

In [ ]:
llm.invoke([
    HumanMessage(content="Hello! My name is Aakash and i am a RAG Consultant"),
    AIMessage(content="Hello, Aakash! Nice to meet you! 🌟 As a RAG Consultant, I'm sure you have a wealth of experience in leveraging Retrieval-Augmented Generation to enhance AI systems."),
    HumanMessage(content="What is my name and what do i do? ")
])

### Message History

In [12]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    # Retrieve the chat history for the given session_id from your storage
    # For example, you can use a database or a file to store the history
    # Here, we'll just return an empty history for demonstration purposes
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(llm,get_session_history)

/home/aakash/anaconda3/envs/ds/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3823: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [13]:
config = {
    "configurable":{"session_id":"Chat1"}
}

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
response = with_message_history.invoke([HumanMessage(content="Hello! My name is Aakash and i am a RAG Consultant")],config=config)

In [ ]:
response

In [ ]:
with_message_history.invoke([
    HumanMessage(content="Whats my name?")],config=config)

In [ ]:
config2 = {
    "configurable":{"session_id":"Chat2"}
}

with_message_history.invoke([
    HumanMessage(content="Whats my name?")],config=config2)

## PROMPT TEMPLATE


In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant,answer all the questions to the best of your ability"),
    MessagesPlaceholder(variable_name="messages"),
])
chain = prompt|llm

In [15]:
chain.invoke({"messages":[
    HumanMessage(content="Hello! My name is Aakash and i am a RAG Consultant")], "session_id":"Chat1"})

AIMessage(content="Hello, Aakash! It's great to meet you. Could you clarify what RAG stands for in your role as a RAG Consultant? I'd love to learn more about your work and how you apply it in your consulting practice. What kind of projects or industries do you typically work with? 😊", additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-10-28T10:26:41.318168824Z', 'done': True, 'done_reason': 'stop', 'total_duration': 59987390943, 'load_duration': 10429685, 'prompt_eval_count': 45, 'prompt_eval_duration': 2788271000, 'eval_count': 295, 'eval_duration': 57152925000, 'logprobs': None, 'model_name': 'qwen3:8b'}, id='run--01a1820b-4dd0-74b3-a9d0-64284285d094-0', usage_metadata={'input_tokens': 45, 'output_tokens': 295, 'total_tokens': 340})

In [16]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)
config = {
    "configurable":{"session_id":"Chat3"}
}
response = with_message_history.invoke([HumanMessage(content="Hello! My name is Aakash and i am a RAG Consultant")],config=config)

/home/aakash/anaconda3/envs/ds/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3823: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [17]:
# Add more complexity

prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant,answer all the questions to the best of your ability in {language}"),
    MessagesPlaceholder(variable_name="messages"),
])
chain = prompt|llm

In [18]:
response = chain.invoke({"messages":[
    HumanMessage(content="Hello! My name is Aakash and i am a RAG Consultant")], "session_id":"Chat4", "language":"Hindi"})

In [19]:
response

AIMessage(content='नमस्ते आकाश! मैं आपके लिए आवश्यकता अनुसार सहायता करने के लिए यहां हूं। आप अपने काम में कैसे मदद कर सकते हैं? आपके पास कोई विशेष प्रश्न या अपेक्षाएं हैं जो आपके RAG काम में सहायता कर सकती हैं? 😊', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-10-28T10:29:15.048158621Z', 'done': True, 'done_reason': 'stop', 'total_duration': 58620948809, 'load_duration': 1348623, 'prompt_eval_count': 47, 'prompt_eval_duration': 1944888000, 'eval_count': 291, 'eval_duration': 56589765000, 'logprobs': None, 'model_name': 'qwen3:8b'}, id='run--01a1820d-aba8-7f31-b4a6-06bc637753da-0', usage_metadata={'input_tokens': 47, 'output_tokens': 291, 'total_tokens': 338})

In [20]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages",configurable_keys=["session_id","language"])

/home/aakash/anaconda3/envs/ds/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3823: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [21]:
config = {
    "configurable":{"session_id":"Chat4"}
}
response = with_message_history.invoke({"messages":[HumanMessage(content="Hello! My name is Aakash and i am a RAG Consultant")],"language":"Hindi"},config=config)

In [22]:
response.content

'नमस्ते अकाश! मैं आपका सहायक हूं, आपके काम में मदद करने के लिए तैयार हूं। आपके राग कonsultant के रूप में काम करने के बारे में बताएं ताकि मैं आपकी सहायता कर सकूं। क्या आपको कोई विशिष्ट जरूरत है या आपको कुछ सलाह चाहिए? 😊'

## MANAGE THE SIZE OF CONVERSATION HISTORY
trim messages helps us reduce how many messages we are sending to the model

In [25]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer = trim_messages(max_tokens=40,strategy="last",token_counter=llm,include_system=True,allow_partial=False,start_on="human")
messages = [
    SystemMessage(content="You are a helpful assistant,answer all the questions to the best of your ability"),
    HumanMessage(content="Hello! My name is Aakash "),
    AIMessage(content="Hello, Aakash! Nice to meet you! 🌟 "),
    HumanMessage(content="I like vanilla ice-cream "),
    AIMessage(content="nice "),
    HumanMessage(content="What is 2+2? "),
    AIMessage(content="4 "),
    HumanMessage(content="thanks "),
    AIMessage(content="No Problem "),
    HumanMessage(content="having fun? "),
    AIMessage(content="Yes "),
                     
]
trimmer.invoke(messages)

[SystemMessage(content='You are a helpful assistant,answer all the questions to the best of your ability', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='No Problem ', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun? ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Yes ', additional_kwargs={}, response_metadata={})]

In [26]:
from operator import itemgetter
from langchain_core.runnables import  RunnablePassthrough

chain = (RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)|prompt|llm)

response=chain.invoke(
            {"messages":messages + [HumanMessage(content="What ice-cream do i like?")],
             "language": "English"}
             )
response.content

'I don’t have personal preferences or experiences, so I can’t know what ice cream you like! 😊 But I’d love to help you explore options. What flavors, textures, or ingredients do you enjoy? (e.g., chocolate, vanilla, fruity, creamy, etc.) Let me know, and I can suggest some ideas! 🍦'

In [27]:

response=chain.invoke(
            {"messages":messages + [HumanMessage(content="What question did i ask?")],
             "language": "English"}
             )
response.content

'The question you asked earlier was "having fun?" 😊'

## LETS WRAP THIS IN MESSAGE HISTORY

In [28]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages",configurable_keys=["session_id","language"])
config = {
    "configurable":{"session_id":"Chat5","language":"English"}
}

/home/aakash/anaconda3/envs/ds/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3823: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [29]:
response = with_message_history.invoke({"messages":[HumanMessage(content="What question did i ask you?")],"language":"English"},config=config)

In [30]:
response.content

'In this conversation, your only message so far is the question you\'re asking: "What question did I ask you?" There is no prior question in this specific interaction. If you\'re referring to a previous conversation or a different context, please provide more details so I can assist you better!'

VECTOR STORES AND RETRIEVERS